# Figure 2 Final Vesion 09-02-26

### removed old figure 3 09-02-26, now replaced by bubble chart.
### revised 08-20-2025 744 to 745 plot marker
### revised 04-08-2025 to have '_' in cell type names in the CLEAN metadata file
### revised 04-08-2025 to run on Bruno
### Uses Nov Version2d object and Ensembl Id TF's, 11-15-24
### added MYF5 to stem cell heatmap

## General Import Functions

In [ ]:
import scanpy as sc
import pandas as pd
import scipy.stats as stats
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import anndata as an

In [ ]:
sc.settings.set_figure_params(dpi=80, dpi_save= 300, fontsize=6, 
                              facecolor='white', frameon=True, 
                              figsize=(3.0, 2.0), vector_friendly = False, 
                              transparent=True, format="pdf")

sns.set_style("ticks", rc = {'text.color' : "black",
                             'linecolor' : "black",
                             'axes.edgecolor' : "black",
                            'axes.labelcolor' : "black",
                            "xtick.color" : "black",
                            "ytick.color" : "black"})
#fig, ax = plt.subplots(figsize = (6.0,6.0))

In [ ]:
# 02-04-2026 added switch to save tau for all genes for work on external paper table.

DATA_IN_MEANS_DIR = '/oak/stanford/groups/quake/jonesbob/TF_TS_Analysis_Final_Manuscript/TFonly/'
DATA_IN_DIR = '/oak/stanford/groups/quake/jonesbob/TF_TS_Analysis_Final_Manuscript/'
DATA_OUT_DIR = '/oak/stanford/groups/quake/jonesbob/TF_TS_Analysis_Final_Manuscript/'  
META_DATA  = 'TSP1_30_version2d_10X_SS_2024_11_12_metadata_CLEAN_TFonly.csv'
COLOR_TABLE = 'CellTypeColorPalettes.csv'
TAU_TABLE = 'TauValuesCelltype.csv'
TFxCT_DATA = 'TSP1_30_by_CellType_Full_Version2d_Nov2024_LogNorm_10XOnly_Mean_TFonly.csv'


# useful if need to find mean expression values for any gene
#DATA_IN_MEANS_DIR = '/oak/stanford/groups/quake/jonesbob/TF_TS_Analsyis_Working_Dir/AllGenes/'
#DATA_OUT_DIR = '/oak/stanford/groups/quake/jonesbob/TF_TS_Analsyis_Working_Dir/AllGenes/'  
#TAU_TABLE = 'TauValuesCelltypeALLGENES.csv'
#TFxCT_DATA = 'TSP1_30_by_CellType_Full_Version2d_Nov2024_LogNorm_10XOnly_Mean_AllGenes.csv'

In [ ]:
# Now read in mean
print(DATA_IN_MEANS_DIR+TFxCT_DATA)
TF_df = pd.read_csv(DATA_IN_MEANS_DIR+TFxCT_DATA, index_col = 0)
TF_df.shape

In [ ]:
# remove zeros and sort for looks
TF_exp_all = TF_df.T
TF_exp = TF_exp_all[TF_exp_all.sum(axis = 1) != 0.0]
TF_exp = TF_exp.sort_index(axis=0).sort_index(axis=1)
TF_exp

In [ ]:

####################################################################
# Compute tau specificity value for each TF
####################################################################
tau_values = pd.Series( index = TF_exp.index)
for gene in TF_exp.index:
    genevector = TF_exp.loc[gene]
    normed = genevector/max(genevector)
    oneminus = 1 - normed
    tau_values.loc[gene] = sum(oneminus) / (len(oneminus) - 1)
tau_values = tau_values.sort_values(ascending=False)

print('Tau Max', tau_values.max())
print('Tau Min', tau_values.min())

Tau_Values = pd.DataFrame(tau_values)
Tau_Values.columns = ['tau']
print(DATA_OUT_DIR+TAU_TABLE)
Tau_Values.to_csv(DATA_OUT_DIR+TAU_TABLE)
display(Tau_Values)
#display(tau_values)

In [ ]:
# a peek at the lowest Tau genes
Tau_Values.iloc[  -30 : , :]

In [ ]:
########## end of tau code #######

In [ ]:
from matplotlib.lines import Line2D

plt.hist(tau_values, bins=50)
plt.axvline(x=0.85, color='black', linestyle='-', linewidth=0.5)
#plt.axvline(x=0.625831, color='black', linestyle='-', linewidth=0.5)

# Custom legend entry without a line
custom_legend = [Line2D([0], [0], linestyle='None', color='black', label='τ < 0.85')]

plt.xlabel("τ Value")
plt.ylabel("Count of τ values")
plt.title("τ values distribution")

#plt.legend(handles=custom_legend, loc='center')
plt.tight_layout()
plt.savefig(DATA_OUT_DIR + 'SuppFig9A_Tau_Distribution.pdf')
plt.show()

In [ ]:
# look at Tf's in cell types 
TF_ctcount =  TF_exp.astype(bool).sum(axis=1).sort_values(ascending = True)
plt.hist(TF_ctcount, bins = 50)
plt.xlabel("Number of Cell types")
plt.ylabel("Number of TF's")
plt.title('Distribution of Transcription Factors across cell types')
TF_ctcount

## Only high tau or low tau

In [ ]:
# break between ubiquitous and specific at 0.85 tau
# reconsider this after quantile normalization

low_tau = list(tau_values.index[tau_values <= 0.85])
hi_tau = list(tau_values.index[tau_values > 0.85])
TF_hi_exp = TF_exp.loc[hi_tau].copy()
TF_low_exp = TF_exp.loc[low_tau].copy()
print(len(TF_low_exp))
print(len((TF_hi_exp)))
print(len((TF_exp)))

In [ ]:
# save list of lowest 100 for figure
low100 = tau_values.sort_values(ascending = True).iloc[0:100]
low100df = pd.DataFrame(low100)
low100df.to_csv(DATA_OUT_DIR+'LowestTauTFs100.csv')
low100df

## Now Make Heatmap. #####

In [ ]:
# read metadata that goes with the full objects
metadata = pd.read_csv(DATA_IN_MEANS_DIR+META_DATA)
print(DATA_IN_MEANS_DIR+META_DATA)
metadata = metadata[metadata.method == '10X']
print(len(metadata.cell_ontology_class.unique()))
metadata

In [ ]:
def PrepColorTable(exp_matrix):

    # map broad cell types to celltypes for those types in exp_matrix
    ct = pd.DataFrame()
    ct['ctypes'] = metadata.cell_ontology_class
    print('length of cell types', len(ct))
    ct['comparts'] = metadata.compartment
    ct['broadct'] = metadata.broad_cell_class
    print('length of cell types', len(ct))
    display(ct)     #one difference here is the ctype names don't have '_' 
    
    ct = ct[ct.ctypes.isin(exp_matrix.columns)]
    ct = ct.drop_duplicates('ctypes')
    print('length of cell types', len(ct))
    newindex = list(ct.ctypes)
    ct.index = newindex
    
    # map colors to the broad cell types
    colorlist = [matplotlib.colors.to_hex(c) for c in plt.cm.tab20.colors] + [matplotlib.colors.to_hex(c) for c in plt.cm.tab20b.colors]
    type_to_comp_dict = dict(zip(ct.broadct.unique(), colorlist))
    ct['col_colors'] = ct.broadct.map(type_to_comp_dict)

    return(ct)

In [ ]:
TF_exp

In [ ]:
TFcolors = PrepColorTable(TF_exp)
print('TF_exp', TF_exp.shape)
TFcolors.to_csv(DATA_IN_DIR+COLOR_TABLE)
display(TFcolors)
ct_order = TFcolors.sort_values('broadct').index 
TFcolors = TFcolors.loc[ct_order]
TFcolors

In [ ]:
ed = TF_exp.T

plot_data = ed.loc[ct_order].T
expr_data_plot = ed.loc[ct_order].T
expr_data_plot  # now the cell types are in order

In [ ]:
TFcolors

In [ ]:
# Make sorting indexes to sort by broad cell type on plot
custom_ct_order_series = pd.Series(data=range(len(ct_order)), index=ct_order, name="order")
custom_ct_order_series.reset_index()

## Heat map all genes all cell types

In [ ]:
sorted_index_df = pd.DataFrame(expr_data_plot.idxmax(axis=1), columns=["index"]).join(custom_ct_order_series, on ="index")
sorted_index_df = sorted_index_df.sort_values(by = "order", kind = 'quicksort')
sorted_index_df

### Note, this sort produces many ties among the indexes.  
### The order of the ties are placed into the dataframe will vary from compute system to compute system.
### Thus the order of genes within each broad cell type will vary in the figure.
### Your heatmaps may look slightly different than the figure in the manuscript but,
### The values will be correct in all cases. 

In [ ]:
Original = True

if Original:
    # Original Sort
    print('Original')
    sorted_index_df = pd.DataFrame(expr_data_plot.idxmax(axis=1), columns=["index"]).join(custom_ct_order_series, on ="index").sort_values(by = "order")
    sorted_index = sorted_index_df.index
    expr_data_plot = expr_data_plot.loc[sorted_index]

else:
    # new sorted within cell types
    print('Gradient')
    peak_df = pd.DataFrame({
        "index": expr_data_plot.idxmax(axis=1),   # cell type where each TF peaks
        "maxexp": expr_data_plot.max(axis=1)      # expression value at that peak
    })

    sorted_index_df = (peak_df
        .join(custom_ct_order_series, on="index")
        .sort_values(by=["order", "maxexp"], ascending=[True, False], kind="mergesort"))

    sorted_index = sorted_index_df.index
    expr_data_plot = expr_data_plot.loc[sorted_index]



In [ ]:
# assign color key on the gene axix by broad cell type
Genecolormapdf = sorted_index_df.reset_index(names="gene").rename(columns = {"index" : "ctypes"}).merge(TFcolors[["ctypes", "broadct", "col_colors"]], on= "ctypes")
Genecolormapdf = Genecolormapdf.set_index("gene")
Genecolormapdf

In [ ]:
# reorder genes in max cell type order
expr_data_plot = expr_data_plot.loc[sorted_index]
expr_data_plot

In [ ]:
# what is the range of this data
print('Max', expr_data_plot.max().max())
print('Min', expr_data_plot.min().min())


In [ ]:
expr_data_plot

In [ ]:
#genrate low rez, all gene Heat Map
# clipped at 1.0

sns.set(font_scale=2)
g = sns.clustermap(expr_data_plot.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               dendrogram_ratio=(.1, .01),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               vmax = np.percentile(expr_data_plot, 99.5),
               #vmin = 0.0,
               row_colors = TFcolors.col_colors,
               col_colors = Genecolormapdf.col_colors,
               yticklabels=False,
               xticklabels=False,
               figsize=(60, 30)
           )

# Move col_colors from top to bottom
# Get the current position of the col_colors axis
#col_colors_ax = g.ax_col_colors
#current_pos = col_colors_ax.get_position()

# Get the heatmap position
#heatmap_pos = g.ax_heatmap.get_position()

# Set new position: move to bottom of heatmap
# Format: [left, bottom, width, height]
#new_pos = [current_pos.x0, 
#           heatmap_pos.y0 - current_pos.height - 0.01,  # Position below heatmap
#           current_pos.width, 
#           current_pos.height]

#col_colors_ax.set_position(new_pos)

plt.savefig(DATA_OUT_DIR + 'Fig2A.pdf')
plt.show()
plt.close()

In [ ]:
# normalize in both directions
def NormTwoWays(exp_input):
    # copy the data 
    df_min_max_scaled = exp_input.copy() 
    
    # apply normalization techniques in one direction.
    for column in df_min_max_scaled.columns: 
        df_min_max_scaled[column] = (df_min_max_scaled[column] - df_min_max_scaled[column].min()) / (df_min_max_scaled[column].max() - df_min_max_scaled[column].min())  

    # copy the data again but flip
    df_min_max_scaled_flip = df_min_max_scaled.T.copy() 

    # apply normalization techniques in other direction.
    for column in df_min_max_scaled_flip.columns: 
        df_min_max_scaled_flip[column] = (df_min_max_scaled_flip[column] - df_min_max_scaled_flip[column].min()) / (df_min_max_scaled_flip[column].max() - df_min_max_scaled_flip[column].min()) 

    # unflip
    normalized_expr_data_plot = df_min_max_scaled_flip.T.copy()
    return(normalized_expr_data_plot)

In [ ]:
# normalize in cell type direction
def NormOneWay(exp_input):
    # copy the data 
    df_min_max_scaled = exp_input.copy().T
    
    # apply normalization techniques in one direction.
    for column in df_min_max_scaled.columns: 
        df_min_max_scaled[column] = (df_min_max_scaled[column] - df_min_max_scaled[column].min()) / (df_min_max_scaled[column].max() - df_min_max_scaled[column].min())  

   
    return(df_min_max_scaled.T)

In [ ]:
#normalized_expr_data_plot = NormTwoWays(expr_data_plot)
normalized_expr_data_plot = NormOneWay(expr_data_plot)
########### Dont Normalize #############
#normalized_expr_data_plot = expr_data_plot
########################################

#genrate low rez, all gene Heat Map
# this one is two-axis normalized and clipped at 1.0
sns.set(font_scale=2)
sns.clustermap(normalized_expr_data_plot.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               dendrogram_ratio=(.1, .01),  #used to position the color bar
               cmap ='Blues',
               vmax = 1.0,
               #vmin = 0.0,
               row_colors = TFcolors.col_colors,
               col_colors = Genecolormapdf.col_colors,
               yticklabels=False,
               xticklabels=False,
               figsize=(60, 30)

               
           )

#plt.savefig(DATA_OUT_DIR + 'Fig2A_LowRezNorm.pdf')
plt.show()
plt.close()


## Make color key

In [ ]:
# copy the color key for the broad cell types into the palette for the key
key_colors = TFcolors[ ['broadct','col_colors']].drop_duplicates()
ct_palette = key_colors.sort_values('broadct')
ct_palette = ct_palette.reset_index()
ct_palette = ct_palette.drop(['index'], axis =1)

ct_palette.to_csv(DATA_OUT_DIR + "BroadCellTypeColorPalette.csv")
ct_palette

In [ ]:
# make fake data for sns.relplot of the correct dimensions and order
fdf = expr_data_plot.T.iloc[0:len(ct_palette)]
fdf = fdf[['LHX5', 'PRRX1']]
fdf['Broad Cell Type'] = list(ct_palette.broadct)
fdf['Color Key'] = list(ct_palette.col_colors)
fdf = fdf.sort_values('Broad Cell Type', ascending = True)
fdf = fdf.reset_index()
fdf = fdf.drop('index', axis = 1)
fdf["Broad Cell Type"] = fdf["Broad Cell Type"].str.capitalize()


In [ ]:
sns.set(font_scale=0.5)
sns.relplot(data=fdf, x="LHX5", y="PRRX1", 
            hue = fdf['Broad Cell Type'], 
            palette = list(fdf['Color Key']))

plt.savefig(DATA_OUT_DIR + 'Fig2A_ColorKey.pdf')
plt.show()
plt.close()


# Make subplots insets for broad cell types 

In [ ]:
# make non-called out genes into ""

tlist = ['ETS1', 'FOXP3', 'IKZF1','IKZF2', 'IKZF3'] 
flist = ['TCF21', 'TBX20', 'SREBF1']
elist = ['ERG','ELK3','HEY1','FOXC2', 'LHX6', 'EPAS1', 'FLI1']
slist = ['ZIC1', 'ZIC4', 'ASCL2', 'MYF5', 'MSC','CDX1', 'CDX2', 'ONECUT2','CBX2', 'HOXA9', 'MYB', 'GATA1', 'KLF1']
mlist = ['SALL4','DMRT1', 'SOHLH1', 'MYBL1','TCFL5', 'SOX30', 'TCF3']
commonlist = ['SOX2', 'OCT4', 'NANOG', 'TBET', 'EOMES', 'MYOD' ]

glist = tlist + flist + elist + slist + mlist + commonlist

In [ ]:
# prune list by tau....
hightaulist = list(tau_values.index[tau_values> 0.85])
print(len(hightaulist))

In [ ]:
high = tau_values[tau_values > 0.85]

In [ ]:
def fixSuplotCellTypeLabels(expm):
    # fix cell type names
    expm.columns = expm.columns.str.capitalize()
    expm.columns = [sub.replace('_', ' ') for sub in subPlotdf.columns] 
    return(expm)

In [ ]:
def fixSuplotGeneLabels(expm, glist):
    flip = expm.T
    geneSelector = flip.columns.isin(glist)
    #print('geneSelector', geneSelector)
    flipgenes = list(flip.columns)
    for i in range(len(geneSelector)):
        if ~geneSelector[i]:
            flipgenes[i] = ' '
    if flipgenes[0] == ' ': 
        flipgenes[0] = '.'
    if flipgenes[-1] == ' ':
        flipgenes[-1] = '.'    
                
    flip.columns = flipgenes
    #print(flipgenes[-1])
    expm = flip.T
    return(expm)

## T-Cell Subplot

In [ ]:
TFcolors.broadct

In [ ]:
subPlotCt = TFcolors[TFcolors.broadct == 't_cell']
subPlotCt

In [ ]:
## T-cell sub figure
subPlotCt = TFcolors[TFcolors.broadct == 't cell']
y = list(subPlotCt.index)
x = Genecolormapdf.ctypes.isin(y)
subPlotdf = expr_data_plot.loc[x, y]

# fix up cell type names
#subPlotdf = fixSuplotCellTypeLabels(subPlotdf)
subPlotdf

In [ ]:
# clipped at 
sns.set(font_scale=6)
sns.clustermap(subPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(120, 60)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotTcell.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdfhigh = fixSuplotGeneLabels(subPlotdf, hightaulist)

# clipped at 
sns.set(font_scale=7)
sns.clustermap(subPlotdfhigh.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(120, 60)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotTcellHigh.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdffav = fixSuplotGeneLabels(subPlotdf, glist)

sns.set(font_scale=7)
sns.clustermap(subPlotdffav.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(120, 60)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotTcellFav.pdf')
plt.show()
plt.close()

## Fibroblast-Cell Subplot

In [ ]:
## Fibroblast sub figure
subPlotCt = TFcolors[TFcolors.broadct == 'fibroblast']
y = list(subPlotCt.index)
x = Genecolormapdf.ctypes.isin(y)
subPlotdf = expr_data_plot.loc[x, y]

# fix up cell type names
subPlotdf = fixSuplotCellTypeLabels(subPlotdf)
subPlotdf

In [ ]:
# clipped at 
sns.set(font_scale=6)
sns.clustermap(subPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(50, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotFibroblast.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdfhigh = fixSuplotGeneLabels(subPlotdf, hightaulist)

# clipped at 
sns.set(font_scale=6)
sns.clustermap(subPlotdfhigh.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(50, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotFibroblastHigh.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdffav = fixSuplotGeneLabels(subPlotdf, glist)

# clipped at 
sns.set(font_scale=6)
sns.clustermap(subPlotdffav.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(50, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotFibroblastFav.pdf')
plt.show()
plt.close()

## Endothelial Cell Subplot

In [ ]:
## Endothelial cell sub figure
subPlotCt = TFcolors[TFcolors.broadct == 'endothelial cell']
y = list(subPlotCt.index)
x = Genecolormapdf.ctypes.isin(y)
subPlotdf = expr_data_plot.loc[x, y]

# fix up cell type names
subPlotdf = fixSuplotCellTypeLabels(subPlotdf)
subPlotdf

In [ ]:
# clipped at 
sns.set(font_scale=7)
sns.clustermap(subPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(60, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotEndo.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdfhigh = fixSuplotGeneLabels(subPlotdf, hightaulist)

# clipped at 
sns.set(font_scale=7)
sns.clustermap(subPlotdfhigh.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(60, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotEndoHigh.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdffav = fixSuplotGeneLabels(subPlotdf, elist)

In [ ]:
# clipped at 
sns.set(font_scale=7)
sns.clustermap(subPlotdffav.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(60, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotEndoFav.pdf')
plt.show()
plt.close()

## Stem Cell Subplot

In [ ]:
## Stem cell sub figure
subPlotCt = TFcolors[TFcolors.broadct == 'stem cell']
y = list(subPlotCt.index)
x = Genecolormapdf.ctypes.isin(y)
subPlotdf = expr_data_plot.loc[x, y]

# fix up cell type names
subPlotdf = fixSuplotCellTypeLabels(subPlotdf)
subPlotdf

In [ ]:
# clipped at 
sns.set(font_scale=5)
sns.clustermap(subPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(90, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotStem.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdfhigh = fixSuplotGeneLabels(subPlotdf, hightaulist)

# clipped at 
sns.set(font_scale=5)
sns.clustermap(subPlotdfhigh.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(90, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotStemHigh.pdf')
plt.show()
plt.close()

In [ ]:
# next few cells are to confirm expression of ZIC1, ZIC4, AND MYF5 
# in fat stem cells as described in the text and figures.

In [ ]:
TF_df.loc[ : , 'MYF5'].sort_values().tail(10)

In [ ]:
TF_df.loc[ : , 'ZIC1'].sort_values().tail(5)

In [ ]:
TF_df.loc[ : , 'ZIC4'].sort_values().tail(5)

In [ ]:
TF_df.loc[ : , 'MYB'].sort_values().tail(10)

In [ ]:
subPlotdfav = fixSuplotGeneLabels(subPlotdf, glist)

# clipped at 
sns.set(font_scale=5)
sns.clustermap(subPlotdfav.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,
               yticklabels=True,
               xticklabels=True,
               figsize=(90, 50)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotStemFav.pdf')
plt.show()
plt.close()

## Male Germ Cell Subplot

In [ ]:
## Male germ cell sub figure
subPlotCt = TFcolors[TFcolors.broadct == 'male germ cell']
y = list(subPlotCt.index)
x = Genecolormapdf.ctypes.isin(y)
subPlotdf = expr_data_plot.loc[x, y]

# fix up cell type names
subPlotdf = fixSuplotCellTypeLabels(subPlotdf)
subPlotdf

In [ ]:
# clipped at 
sns.set(font_scale=4)
sns.clustermap(subPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(200, 40)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotMale.pdf')
plt.show()
plt.close()

In [ ]:
subPlotdf.loc['SOX30', :]

In [ ]:
# save for later tf lookup
malectTfDf = subPlotdf

In [ ]:
#look up some genes
mbtgs = len(malectTfDf.index)
print('How many TF in male germ subplot?', mbtgs)
mbtgsTau = tau_values[malectTfDf.index]
specTaumg = mbtgsTau[mbtgsTau > 0.85]
print('How many are Tau >0.85?',len(specTaumg))
print('Percent  Tau >0.85?',len(specTaumg)/mbtgs*100)
TFofInterest = 'DMRT1'
print('Tau of', TFofInterest, '=', mbtgsTau.loc[TFofInterest] )

Zee = 'Z'
zincs = [idx for idx in list(malectTfDf.index) if idx.startswith(Zee)]
print('Number of Zinc Finger all male germ cell TFs', len(zincs))
print('Percent Zinc Finger of all male germ cell TFs', len(zincs)/mbtgs*100)


zincs = [idx for idx in list(specTaumg.index) if idx.startswith(Zee)]
print('Number of Zinc Finger SPECIFIC male germ cell TFs', len(zincs))
print('Percent  Zinc Finger SPECIFIC of SPECIFIC male germ cell TFs', len(zincs)/len(specTaumg)*100)

In [ ]:
subPlotdfhigh = fixSuplotGeneLabels(subPlotdf, hightaulist)

# clipped at 
sns.set(font_scale=4)
sns.clustermap(subPlotdfhigh.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(200, 40)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotMaleHigh.pdf')
plt.show()
plt.close()

In [ ]:
TF_df.loc[ : , 'SALL4'].sort_values().tail(10)

In [ ]:
TF_df.loc[ : , 'MYBL1'].sort_values().tail(10)

In [ ]:
subPlotdffav = fixSuplotGeneLabels(subPlotdf, glist)

# clipped at 
sns.set(font_scale=4)
sns.clustermap(subPlotdffav.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(200, 40)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubPlotMaleFav.pdf')
plt.show()
plt.close()

In [ ]:
# broken up version for supllementaal figure not
y = list(subPlotCt.index)
y

In [ ]:
x = Genecolormapdf.ctypes.isin(['spermatogonium'])
GoniumSubPlotdf = expr_data_plot.loc[x, y]

x = Genecolormapdf.ctypes.isin(['male_germ_cell'])
MGCSubPlotdf = expr_data_plot.loc[x, y]

x = Genecolormapdf.ctypes.isin(['spermatid'])
StidSubPlotdf = expr_data_plot.loc[x, y]

x = Genecolormapdf.ctypes.isin(['spermatocyte'])
ScyteSubPlotdf = expr_data_plot.loc[x, y]



In [ ]:
print(GoniumSubPlotdf.shape)
xdim = GoniumSubPlotdf.shape[0]
sns.set(font_scale=4)
sns.clustermap(GoniumSubPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(xdim, 20)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubGonium.pdf')
plt.show()
plt.close()

In [ ]:
print(MGCSubPlotdf.shape)
xdim = MGCSubPlotdf.shape[0]
sns.set(font_scale=4)
sns.clustermap(MGCSubPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(xdim, 20)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubMGC.pdf')
plt.show()
plt.close()

In [ ]:
print(StidSubPlotdf.shape)
xdim = MGCSubPlotdf.shape[0]
sns.set(font_scale=4)
sns.clustermap(StidSubPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(xdim, 20)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubStid.pdf')
plt.show()
plt.close()

In [ ]:
print(ScyteSubPlotdf.shape)
xdim = ScyteSubPlotdf.shape[0]
sns.set(font_scale=4)
sns.clustermap(ScyteSubPlotdf.T, 
               row_cluster=False, 
               col_cluster=False,
               #optimal_ordering=True,
               #dendrogram_ratio=(.2, .1),  #used to position the color bar
               cmap ='Blues',
               #vmax = 1.0,0
               #vmin = 0.0,
               #row_colors = TFcolors.col_colors,
               #col_colors = Genecolormapdf.col_colors,               
               yticklabels=True,
               xticklabels=True,
               figsize=(xdim, 20)
           )
plt.savefig(DATA_OUT_DIR + 'Fig2SubScyte.pdf')
plt.show()
plt.close()